# echoSentinel v2 — Train the PANNs+PCEN engine on a free GPU

Runs on **Google Colab** (Runtime -> Change runtime type -> **T4 GPU**) or Kaggle.

Steps: upload the repo + Dataset, install, fetch the pretrained CNN14 backbone once,
train, then download the trained weights back to your machine into `weights/`.

Checkpointing is per-epoch on best val F1, so a Colab disconnect never loses the best model.

## 1. Get the code and data onto the machine

**Option A — Google Drive (recommended).** Zip your local `echosentinel_v2/` folder
*including* `Dataset/`, upload the zip to Drive, then:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Adjust the path to where you put the zip in your Drive:
!cp '/content/drive/MyDrive/echosentinel_v2.zip' /content/
!cd /content && unzip -q -o echosentinel_v2.zip
%cd /content/echosentinel_v2

**Option B — GitHub.** If you push the repo (Dataset is gitignored, so upload data separately):
```python
!git clone https://github.com/<you>/echosentinel_v2.git
%cd echosentinel_v2
# then upload Dataset/ via Drive as in Option A
```

## 2. Install the package

In [ ]:
!pip install -q -e .
import torch; print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Build manifests (if not already present) and fetch the pretrained backbone

The audit CSV ships in the repo; rebuild the manifest in case paths differ, then
download the CNN14 AudioSet checkpoint once (~310 MB).

In [ ]:
!python scripts/00_audit_dataset.py
!python scripts/01_build_manifests.py
!python scripts/fetch_pretrained.py

## 4. Train PANNs + PCEN

~60 epochs on a T4 is roughly 1–2 hours. Lower `--epochs` for a quick run.
Best-by-val-F1 weights are written to `weights/panns_pcen.pt`.

In [ ]:
!python scripts/04_train.py --config configs/model_panns.yaml

## 5. Evaluate locally with the exact competition metric (IER)

Build a synthetic validation set that mirrors the noisy test conditions, run the
trained model, and score IER.

In [ ]:
!python scripts/03_build_synth_valset.py --n-scenes 24 --seconds 60
!python predict.py --input_dir out/synth_valset --output_json out/results.json --weights weights/panns_pcen.pt
!python scripts/05_evaluate_ier.py --predictions out/results.json --ground-truth out/synth_valset/ground_truth.json

## 6. Save the trained weights back to Drive

Then download `weights/panns_pcen.pt` from Drive into your local repo's `weights/`
folder — inference/Docker will use it offline.

In [ ]:
!cp weights/panns_pcen.pt '/content/drive/MyDrive/panns_pcen.pt'
print('Saved trained weights to Drive. Download it into your local weights/ folder.')